# Traffic Signal Reinforcement Learning — Reproducibility Notebook

This notebook is the single entry point for reproducing our project. It sets up the environment, verifies the repo structure, installs dependencies, builds/installs CityFlow if needed, runs Q-learning, Stable-Baselines experiments, the DQN+LSTM pipeline, and LSTM vector experiments, and then summarizes the generated results.

Run the below cells from top to bottom. Make sure you have created a venv beforehand following Step 1 in README.md, and select that venv as the interpreter.

## 1. Locate the project root

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("Project root:", PROJECT_ROOT)
print("PYTHONPATH set")

Project root: /Users/achakrav6/Desktop/lab4/testing-asi
PYTHONPATH set


In [2]:
from pathlib import Path
import os, sys, subprocess, json, textwrap, shutil, glob

NOTEBOOK_START = Path.cwd().resolve()

def find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "README.md").exists() and (p / "env" / "cityflow_env.py").exists() and (p / "scripts").exists():
            return p
    for p in start.glob("*"):
        if p.is_dir() and (p / "README.md").exists() and (p / "env" / "cityflow_env.py").exists():
            return p.resolve()
    raise FileNotFoundError("Could not find ASI-PROJECT root. Open this notebook from inside the project folder.")

PROJECT_ROOT = find_project_root(NOTEBOOK_START)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print("Python:", sys.executable)

Project root: /Users/achakrav6/Desktop/lab4/testing-asi
Python: /Users/achakrav6/Desktop/lab4/testing-asi/venv/bin/python


## 2. Verify expected repo structure

The project should contain the simulator wrapper, scripts, data, results folders, and requirements file.

In [3]:
expected_paths = [
    "env/cityflow_env.py",
    "scripts/run_simulation.py",
    "scripts/baseline",
    "scripts/q_learning_experiments",
    "scripts/stable_baselines",
    "scripts/dqn_lstm",
    "scripts/lstm_vector",
    "data",
    "results",
    "requirements.txt",
]

missing = [p for p in expected_paths if not Path(p).exists()]
if missing:
    print("Missing expected paths:")
    for p in missing:
        print(" -", p)
else:
    print("All expected project paths found.")

print("\nTop-level folders/files:")
for p in sorted(Path('.').iterdir()):
    print("" if p.is_dir() else "", p.name)

All expected project paths found.

Top-level folders/files:
 .DS_Store
 .git
 .gitignore
 CityFlow
 README.md
 Traffic_Signal_RL_Reproducibility.ipynb
 data
 env
 figures
 hangzhou_datasets
 logs
 models
 output.log
 requirements.txt
 results
 scripts
 venv


## 3. Method Definition

In [17]:
import subprocess

def run(cmd, check=True, env=None):
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        env=merged_env,
        check=check
    )
    if proc.returncode != 0:
        print(f"Command failed with exit code {proc.returncode}")
    return proc

def run_python_script(path, args=None, required=False):
    path = Path(path)
    args = args or []

    if not path.exists():
        msg = f"Skipping missing script: {path}"
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return None

    cmd = [sys.executable, str(path), *args]
    print("$", " ".join(map(str, cmd)))

    proc = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        env=os.environ.copy(),
        check=required,
    )

    if proc.returncode != 0:
        print(f"Command failed with exit code {proc.returncode}")
    return proc

## 3a. Install Python dependencies (`requirements.txt`)

Same order as `README.md`:

1. `pip install -r requirements.txt`
2. `python -m pip install --upgrade pip setuptools wheel`



In [5]:
requirements_file = PROJECT_ROOT / "requirements.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Missing {requirements_file}")

print("Installing packages from requirements.txt (README step 1)...")
run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
    check=False,
)

print("Upgrading pip, setuptools, wheel (README step 2)...")
run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    check=False,
)



Installing packages from requirements.txt (README step 1)...
$ /Users/achakrav6/Desktop/lab4/testing-asi/venv/bin/python -m pip install -r /Users/achakrav6/Desktop/lab4/testing-asi/requirements.txt
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1
Upgrading pip, setuptools, wheel (README step 2)...
$ /Users/achakrav6/Desktop/lab4/testing-asi/venv/bin/python -m pip install --upgrade pip setuptools wheel
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


CompletedProcess(args=['/Users/achakrav6/Desktop/lab4/testing-asi/venv/bin/python', '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], returncode=0)

## 4. Build/install CityFlow if needed

This matches the **CityFlow** section in `README.md` (clone → submodules → optional pybind11 → patch `cmake_minimum_required` → `pip install` → optional `CMAKE_ARGS`).

- If `import cityflow` already works, the cell does nothing.
- If `CityFlow/` is missing, it **clones** the official repo (requires **git** and **network**).
- If the folder is a git checkout, it runs **`git submodule update --init --recursive`**.
- It patches old **`cmake_minimum_required(...)`** values in `CMakeLists.txt` and the pybind11 helper paths listed in the README.
- It runs **`pip install -e CityFlow/`** when `setup.py` exists (same as editable local install; otherwise a regular `pip install` of that directory).
- If the build still fails, it retries with **`CMAKE_ARGS="-DCMAKE_POLICY_VERSION_MINIMUM=3.5"`** (README), then as a last resort replaces **`extern/pybind11`** with upstream pybind11 and tries again.


In [6]:
import importlib.util
import re
import shutil

cityflow_installed = importlib.util.find_spec("cityflow") is not None
print("CityFlow import works before build cell:", cityflow_installed)

cityflow_dir = PROJECT_ROOT / "CityFlow"


def patch_cmake_minimum_files(root: Path) -> list[Path]:
    """Match README: bump old cmake_minimum_required to 3.5 in CityFlow and pybind11 cmake files."""
    extra = [
        root / "extern" / "pybind11" / "CMakeLists.txt",
        root / "extern" / "pybind11" / "tools" / "pybind11Tools.cmake",
    ]
    seen: set[Path] = set()
    patched: list[Path] = []
    for path in list(root.rglob("CMakeLists.txt")) + extra:
        if not path.is_file():
            continue
        key = path.resolve()
        if key in seen:
            continue
        seen.add(key)
        text = path.read_text(encoding="utf-8", errors="ignore")
        new_text = re.sub(
            r"cmake_minimum_required\s*\(\s*VERSION\s+[0-9.]+\s*\)",
            "cmake_minimum_required(VERSION 3.5)",
            text,
        )
        if new_text != text:
            path.write_text(new_text, encoding="utf-8")
            patched.append(path)
            print("Patched cmake_minimum_required in:", path)
    return patched


def pip_install_cityflow(editable: bool):
    cmd = [sys.executable, "-m", "pip", "install"]
    if editable:
        cmd.append("-e")
    cmd.append(str(cityflow_dir))
    return run(cmd, check=False)


def pip_install_cityflow_with_cmake_policy(editable: bool):
    cmd = [sys.executable, "-m", "pip", "install"]
    if editable:
        cmd.append("-e")
    cmd.append(str(cityflow_dir))
    return run(
        cmd,
        check=False,
        env={"CMAKE_ARGS": "-DCMAKE_POLICY_VERSION_MINIMUM=3.5"},
    )


def replace_pybind11_with_upstream() -> None:
    pb = cityflow_dir / "extern" / "pybind11"
    if not pb.exists():
        print("No extern/pybind11 present; skipping pybind11 replacement.")
        return
    print("README fix: removing bundled extern/pybind11 and cloning upstream pybind11 ...")
    shutil.rmtree(pb)
    run(
        [
            "git",
            "clone",
            "https://github.com/pybind/pybind11.git",
            str(pb),
        ],
        check=True,
    )


if cityflow_installed:
    print("Skipping CityFlow setup because `import cityflow` already works.")
else:
    if not cityflow_dir.exists():
        print("Cloning CityFlow (needs git + network), same as README ...")
        run(
            [
                "git",
                "clone",
                "https://github.com/cityflow-project/CityFlow.git",
                str(cityflow_dir),
            ],
            check=True,
        )

    if (cityflow_dir / ".git").exists():
        print("git submodule update --init --recursive (README) ...")
        run(
            ["git", "-C", str(cityflow_dir), "submodule", "update", "--init", "--recursive"],
            check=False,
        )
    else:
        print("No .git under CityFlow/; skipping submodule update (use a full git clone for submodules).")

    patch_cmake_minimum_files(cityflow_dir)

    setup_py = cityflow_dir / "setup.py"
    editable = setup_py.exists()
    print("pip install CityFlow (editable=" + str(editable) + ", same as `pip install .` from CityFlow dir) ...")
    proc = pip_install_cityflow(editable=editable)
    if proc.returncode != 0:
        print("Retrying with CMAKE_ARGS=-DCMAKE_POLICY_VERSION_MINIMUM=3.5 (README) ...")
        proc = pip_install_cityflow_with_cmake_policy(editable=editable)

    if proc.returncode != 0:
        print("Still failing; trying upstream pybind11 (README) then rebuild ...")
        replace_pybind11_with_upstream()
        patch_cmake_minimum_files(cityflow_dir)
        proc = pip_install_cityflow(editable=editable)
        if proc.returncode != 0:
            proc = pip_install_cityflow_with_cmake_policy(editable=editable)

    if proc.returncode != 0:
        raise RuntimeError(
            "CityFlow build/install failed after README-style retries. "
            "Check compiler/CMake, or run the manual commands in README.md from a terminal."
        )
    print("CityFlow pip install finished OK. Restart the kernel, then run the next import cell.")


CityFlow import works before build cell: True
Skipping CityFlow setup because `import cityflow` already works.


If you just installed cityflow above, make sure to restart the kernel before proceeding. You must restart the kernel before attempting to import cityflow. Make sure the below block functions.

In [7]:
import cityflow
print("CityFlow installed successfully")
print("cityflow module:", cityflow)


CityFlow installed successfully
cityflow module: <module 'cityflow' from '/Users/achakrav6/Desktop/lab4/testing-asi/CityFlow/cityflow.cpython-313-darwin.so'>


## 5. Create output folders

All runs write outputs into `results/`, `figures/`, `logs/`, and `models/`.

In [8]:
for folder in ["results", "figures", "logs", "models"]:
    Path(folder).mkdir(exist_ok=True)
print("Output folders ready.")

Output folders ready.


## 6. Quick smoke test: import environment

This verifies that the custom Gymnasium wrapper can be imported before running experiments.

In [9]:
from env.cityflow_env import CityFlowEnv
print("Imported CityFlowEnv successfully:", CityFlowEnv)

Imported CityFlowEnv successfully: <class 'env.cityflow_env.CityFlowEnv'>


## 7. Locate available runnable scripts

Because grading environments can differ, this cell lists the project scripts and uses subprocess calls rather than rewriting experiment logic inside the notebook.

In [10]:
script_files = sorted(Path("scripts").rglob("*.py"))
for s in script_files:
    print(s)

scripts/baseline/plot_baseline_results.py
scripts/baseline/run_baseline.py
scripts/dqn_lstm/collect_lstm_data.py
scripts/dqn_lstm/train.py
scripts/dqn_lstm/train_dqn_with_lstm.py
scripts/dqn_lstm/train_lstm.py
scripts/lstm_vector/lstm_vector_train.py
scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_default_reward.py
scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_delta_wait.py
scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_wait_small_vehicle.py
scripts/q_learning_experiments/bucketed_no_phase/train_bucketed_no_phase_wait_vehicle.py
scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_default_reward.py
scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_delta_wait.py
scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_wait_small_vehicle.py
scripts/q_learning_experiments/bucketed_with_phase/train_bucketed_with_phase_wait_vehicle.py
scripts/q

## 8. Run Q-learning experiments

This is the main tabular Q-learning portion of the project.


In [11]:
import shutil
from pathlib import Path

cityflow_dir = PROJECT_ROOT / "CityFlow"
target_dir = cityflow_dir / "hangzhou_datasets"

candidate_sources = [
    PROJECT_ROOT / "hangzhou_datasets"
]

if target_dir.exists():
    print(f"Dataset target already exists: {target_dir}")
else:
    src_dir = next((p for p in candidate_sources if p.exists() and p.is_dir()), None)
    if src_dir is None:
        print("Could not find source hangzhou_datasets folder.")
        print("Checked:")
        for p in candidate_sources:
            print(f" - {p}")
        print("Place the dataset folder in one of the locations above, then rerun this cell.")
    else:
        print(f"Copying datasets from {src_dir} -> {target_dir}")
        shutil.copytree(src_dir, target_dir)
        print("Copied hangzhou_datasets successfully.")

Copying datasets from /Users/achakrav6/Desktop/lab4/testing-asi/hangzhou_datasets -> /Users/achakrav6/Desktop/lab4/testing-asi/CityFlow/hangzhou_datasets
Copied hangzhou_datasets successfully.


In [ ]:
from pathlib import Path

q_root = Path("scripts/q_learning_experiments")
all_train_scripts = sorted(q_root.glob("**/train_*.py"))
compare_script = q_root / "compare_hangzhou_models.py"

if all_train_scripts:
    for script in all_train_scripts:
        print(f"Running Q-learning script: {script}")
        run_python_script(script, required=False)
else:
    print("No Q-learning train scripts found.")

if compare_script.exists():
    print(f"Running comparison script: {compare_script}")
    run_python_script(compare_script, required=False)

## 9. Run Stable-Baselines experiments

This runs DQN/PPO/A2C comparison scripts if available.


In [ ]:
sb3_candidates = [
    "scripts/stable_baselines/train_dqn.py",
    "scripts/stable_baselines/train_ppo.py",
    "scripts/stable_baselines/train_a2c.py",
    "scripts/stable_baselines/compare_sb3_models.py",
]

ran_any = False
for candidate in sb3_candidates:
    if Path(candidate).exists():
        print(f"Running Stable-Baselines candidate: {candidate}")
        run_python_script(candidate, required=False)
        ran_any = True

if not ran_any:
    print("No Stable-Baselines script found among expected candidates.")

## 10. Run DQN + LSTM pipeline

This section runs the LSTM data collection, LSTM training, and DQN-with-LSTM training scripts if they are available.


In [ ]:
dqn_lstm_steps = [
    "scripts/dqn_lstm/collect_lstm_data.py",
    "scripts/dqn_lstm/train_lstm.py",
    "scripts/dqn_lstm/train_dqn_with_lstm.py",
]

for step in dqn_lstm_steps:
    run_python_script(step, required=False)

## 11. Run LSTM vector experiments

This section runs the vector-based LSTM training script if it is available in the repository.


In [ ]:
lstm_vector_candidates = [
    "scripts/lstm_vector/lstm_vector_train.py",
]

ran_any = False
for candidate in lstm_vector_candidates:
    if Path(candidate).exists():
        print(f"Running LSTM-vector candidate: {candidate}")
        run_python_script(candidate, required=False)
        ran_any = True

if not ran_any:
    print("No LSTM-vector script found among expected candidates.")

## 12. Load and summarize result files

This cell reads any CSV files generated in `results/` and displays their final rows.

In [ ]:
import pandas as pd
from IPython.display import display

csv_files = sorted(Path("results").rglob("*.csv"))
print(f"Found {len(csv_files)} CSV result files.")

if not csv_files:
    print("No CSV files found yet. Check the script output above or confirm that scripts write to results/.")
else:
    for csv in csv_files:
        print("\n===", csv, "===")
        try:
            df = pd.read_csv(csv)
            print("shape:", df.shape)
            display(df.tail(10))
        except Exception as e:
            print("Could not read CSV:", e)

## 13. Show generated figures

This displays figures from the project `figures/` folder, including plots produced by project scripts and by the previous cell.

In [ ]:
from IPython.display import Image, display

fig_files = sorted(list(Path("figures").rglob("*.png")) + list(Path("figures").rglob("*.jpg")) + list(Path("figures").rglob("*.jpeg")))
print(f"Found {len(fig_files)} figure files.")

for fig in fig_files[:20]:
    print("\n", fig)
    display(Image(filename=str(fig)))

## 14. Final reproducibility checklist

By the end of this notebook, you should be able to verify:

1. Python dependencies installed successfully.
2. CityFlow imported successfully.
3. The custom `CityFlowEnv` imported successfully.
4. Q-learning, Stable-Baselines, DQN+LSTM, and LSTM-vector scripts were completed from the repo.
5. Results were written to `results/`.

If a script cell is skipped, check that the script filename in the repo matches the candidate list in that section.